In [7]:
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib as plt

In [109]:
#extracting ticker data and putting it into a pandas df, then extracting only close prices
with open("ftse100current.txt") as f:
    tickers = [line.strip() for line in f]

print(tickers)
print(len(tickers))

data = yf.download(tickers, start = "2015-01-01", end = "2026-08-01", auto_adjust=True)
prices = data["Close"]
monthly_prices = prices.resample("ME").last()

['AAF.L', 'AAL.L', 'ABDN.L', 'ABF.L', 'ADM.L', 'ALW.L', 'ANTO.L', 'AUTO.L', 'AV.L', 'AZN.L', 'BA.L', 'BAB.L', 'BARC.L', 'BATS.L', 'BBOX.L', 'BEZ.L', 'BGEO.L', 'BLND.L', 'BNZL.L', 'BP.L', 'BRBY.L', 'BT-A.L', 'BTRW.L', 'CCC.L', 'CCEP.L', 'CCH.L', 'CNA.L', 'CPG.L', 'CRDA.L', 'CTEC.L', 'DCC.L', 'DGE.L', 'DPLM.L', 'EDV.L', 'ENT.L', 'EXPN.L', 'FCIT.L', 'FRES.L', 'GAW.L', 'GLEN.L', 'GSK.L', 'HLMA.L', 'HLN.L', 'HSBA.L', 'HSX.L', 'HWDN.L', 'IAG.L', 'ICG.L', 'IGG.L', 'IHG.L', 'III.L', 'IMB.L', 'IMI.L', 'INF.L', 'INVP.L', 'ITRK.L', 'JD.L', 'KGF.L', 'LAND.L', 'LGEN.L', 'LLOY.L', 'LMP.L', 'LSEG.L', 'MKS.L', 'MNG.L', 'MRO.L', 'MTLN.L', 'NG.L', 'NWG.L', 'NXT.L', 'PCT.L', 'PRU.L', 'PSH.L', 'PSN.L', 'PSON.L', 'REL.L', 'RIO.L', 'RKT.L', 'RR.L', 'RTO.L', 'SBRY.L', 'SDLF.L', 'SDR.L', 'SGE.L', 'SGRO.L', 'SHEL.L', 'SMIN.L', 'SMT.L', 'SN.L', 'SPX.L', 'SSE.L', 'STAN.L', 'STJ.L', 'SVT.L', 'TSCO.L', 'ULVR.L', 'UU.L', 'VOD.L', 'WEIR.L', 'WTB.L']
100


[*********************100%***********************]  100 of 100 completed


In [102]:
#checking for duplicates
print(monthly_prices.index.duplicated().sum())
print(monthly_prices.columns.duplicated().sum())
#checking for negative prices
for t in monthly_prices.columns:
    if (monthly_prices[t] <=0).sum() > 0:
        print(t)

0
0


In [110]:
#exploring missing values
missing_tickers=[]
for t in prices.columns:
    NaN_count = monthly_prices[t].isna().sum()
    if NaN_count > 0:
        print(t, NaN_count)
        missing_tickers.append(t)

for t in missing_tickers:
    print(t, monthly_prices[t].first_valid_index(), monthly_prices[t].last_valid_index())

AAF.L 53
AUTO.L 2
CCEP.L 50
CTEC.L 21
EDV.L 36
HLN.L 90
MNG.L 57
MTLN.L 127
PSH.L 26
AAF.L 2019-06-30 00:00:00 2026-07-31 00:00:00
AUTO.L 2015-03-31 00:00:00 2026-07-31 00:00:00
CCEP.L 2019-03-31 00:00:00 2026-07-31 00:00:00
CTEC.L 2016-10-31 00:00:00 2026-07-31 00:00:00
EDV.L 2018-01-31 00:00:00 2026-07-31 00:00:00
HLN.L 2022-07-31 00:00:00 2026-07-31 00:00:00
MNG.L 2019-10-31 00:00:00 2026-07-31 00:00:00
MTLN.L 2025-08-31 00:00:00 2026-07-31 00:00:00
PSH.L 2017-03-31 00:00:00 2026-07-31 00:00:00


In [ ]:
#AAF - IPO'ed June 28 2019, no NaN from 2019-06
#AUTO - IPO'ed March 19 2015 , no Nan from 2015-03
#CCEP - listed on LSE March 2019, no NaN from 2019-03
#CTEC - IPO from October 2016, no NaN from 2016-10
#EDV - listed on LSE June 2021 - pre june 2021 data is from toronto exchange so exclude
#HLN - listed june 2022 following demerger - no NaN from 2022-07-31
#MNG - listed Oct 2019 following demerger - no NaN from 2019-10-31
#MTLN - listed Aug 2025 - no NaN from 2025-08 but will be too late for this strategy anyways
#PSH - listed may 2017 - no Nan from then - but exclude the march and april data
monthly_prices["PSH.L"].loc["2017":"2026"]

Date
2017-01-31            NaN
2017-02-28            NaN
2017-03-31            NaN
2017-04-30            NaN
2017-05-31    1206.397217
                 ...     
2026-03-31    3903.642090
2026-04-30    4089.625000
2026-05-31    4119.805664
2026-06-30    3729.824219
2026-07-31    3807.820557
Freq: ME, Name: PSH.L, Length: 115, dtype: float64

In [122]:
#excluding EDV and PSH non-LSE data
monthly_prices.loc[monthly_prices.index < "2017-05-31", "PSH.L"] = np.nan
monthly_prices.loc[monthly_prices.index < "2021-06-30", "EDV.L"] = np.nan

monthly_prices["PSH.L"].loc["2016":"2017"]
monthly_prices["EDV.L"].loc["2020":"2021"]

Date
2020-01-31            NaN
2020-02-29            NaN
2020-03-31            NaN
2020-04-30            NaN
2020-05-31            NaN
2020-06-30            NaN
2020-07-31            NaN
2020-08-31            NaN
2020-09-30            NaN
2020-10-31            NaN
2020-11-30            NaN
2020-12-31            NaN
2021-01-31            NaN
2021-02-28            NaN
2021-03-31            NaN
2021-04-30            NaN
2021-05-31            NaN
2021-06-30    1546.394775
2021-07-31    1711.010864
2021-08-31    1760.894775
2021-09-30    1706.294067
2021-10-31    1850.979980
2021-11-30    1771.153320
2021-12-31    1666.380737
Freq: ME, Name: EDV.L, dtype: float64

In [137]:
#checking for large swings - could be acquisition, stock split
r = monthly_prices.pct_change()
for t in r.columns:
    if (r[t].abs() > 0.5).any():
        print(t)

AAF.L
AAL.L
BAB.L
FRES.L
IAG.L
INVP.L
MRO.L
RR.L


In [ ]:
for i in r.index:
    if abs(r["AAF.L"].loc[i]) > 0.5:
        print(i)
monthly_prices["AAF.L"].loc["2020"]
#research showed it was just affected by covid badly

for i in r.index:
    if abs(r["AAL.L"].loc[i]) > 0.5:
        print(i)
monthly_prices["AAL.L"].loc["2016"]
#research shows just heavilty affected by commodity price collapse

for i in r.index:
    if abs(r["BAB.L"].loc[i]) > 0.5:
        print(i)
monthly_prices["BAB.L"].loc["2020"]
#research indicates no acquisitin/split

for i in r.index:
    if abs(r["FRES.L"].loc[i]) > 0.5:
        print(i)
monthly_prices["FRES.L"].loc["2016"]
#research indicates no acquisitin/split

for i in r.index:
    if abs(r["IAG.L"].loc[i]) > 0.5:
        print(i)
monthly_prices["IAG.L"].loc["2020"]
#airline company so was affected by covid 

for i in r.index:
    if abs(r["INVP.L"].loc[i]) > 0.5:
        print(i)
monthly_prices["INVP.L"].loc["2020"]
#research indicates no acquisitin/split


2020-06-30 00:00:00
2016-02-29 00:00:00
2020-11-30 00:00:00
2016-06-30 00:00:00
2020-03-31 00:00:00
2020-09-30 00:00:00
2020-11-30 00:00:00
2020-03-31 00:00:00


Date
2020-01-31    304.487823
2020-02-29    284.803497
2020-03-31    141.863617
2020-04-30    153.117493
2020-05-31    136.166702
2020-06-30    150.829422
2020-07-31    140.556137
2020-08-31    137.614319
2020-09-30    133.364960
2020-10-31    133.878677
2020-11-30    173.337051
2020-12-31    174.976456
2021-01-31    177.825729
2021-02-28    182.870407
2021-03-31    204.777542
2021-04-30    272.320526
2021-05-31    289.883606
2021-06-30    269.331085
2021-07-31    255.670792
2021-08-31    285.854156
2021-09-30    298.095612
2021-10-31    310.056824
2021-11-30    351.360443
2021-12-31    376.234558
Freq: ME, Name: INVP.L, dtype: float64